In [5]:
import requests
import io
from ultralytics import YOLO
from transformers import pipeline
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np # Add numpy for image processing
print("✅ Multi-Modal Setup Complete!")

✅ Multi-Modal Setup Complete!


# Day 20: Multi-Modal AI - Text-Guided Object Detection

## Key New Concepts Learned Today

**Note 1: What is Multi-Modal AI?**
- Combining different types of data: **Image + Text**
- Example: You type "red car" or "person wearing mask" → model finds it

**Note 2: Our Approach (Advanced)**
- YOLO detects possible objects (fast)
- CLIP (OpenAI) understands text and matches it with image regions
- Result: You can detect **anything** by typing natural language

**Note 3: Why This Is Powerful?**
- No need to retrain model for new classes
- Very flexible and modern (2026 standard)
- Highly valued in research, startups, and hackathons

In [ ]:
# Load models
yolo_model = YOLO("yolo11s.pt")
clip_model = pipeline("zero-shot-image-classification",
                      model="openai/clip-vit-large-patch14")

def text_guided_detection(image_path, text_queries):
    """
    Test function for Text-Guided Detection
    """
    # Load image
    if isinstance(image_path, str):
        if image_path.startswith('http://') or image_path.startswith('https://'):
            response = requests.get(image_path)
            image = Image.open(io.BytesIO(response.content))
        else:
            image = Image.open(image_path)
        image_np = np.array(image)
    else:
        image_np = np.array(image_path)

    # Step 1: YOLO detects regions
    results = yolo_model(image_np, conf=0.3, verbose=False)

    print(f"🔍 Searching for: {text_queries}\n")

    for i, box in enumerate(results[0].boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cropped = image_np[y1:y2, x1:x2]

        # Step 2: CLIP matches with text
        result = clip_model(Image.fromarray(cropped), text_queries)

        best_match = result[0]
        print(f"Object {i+1}: {best_match['label']} ({best_match['score']:.3f})")

    # Show final result
    annotated = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated)
    plt.title("Text-Guided Detection Result")
    plt.axis('off')
    plt.show()

# ====================== TEST IT HERE ======================

# Test 1: Bus image with custom text
text_guided_detection("https://ultralytics.com/images/bus.jpg",
                      ["person", "bus", "traffic light", "bag", "phone"])

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

🔍 Searching for: ['person', 'bus', 'traffic light', 'bag', 'phone']

Object 1: bus (0.998)
